## Transform Refunds Data

1. Extract specific portion of the string from `refund_reason` using `split` function
2. Extract specific portion of the string from `refund_reason` using `regexp_extract` function
3. Extract date and time from the `refund_timestamp`
4. Write transformed data to the Silver schema in hive metastore //not possible here as we do not have hive metastore, we only have unity catalog.
5. I am able to put the data from external location to unity catalog, I think previosly it was not possible, as FOREIGN CATALOG was not available at that time.

In [0]:
%sql
select * from awssql_gizmobox_refund_db_sql_catalog.public.refunds

In [0]:
%sql
SELECT
  refund_id,
  refund_timestamp,
  refund_amount,
  split(refund_reason, ':')[0] as refund_reason,
  split(refund_reason, ':')[1] as refund_source
FROM
  awssql_gizmobox_refund_db_sql_catalog.public.refunds

In [0]:
%sql
SELECT
  refund_id,
  payment_id,
  DATE(refund_timestamp) AS refund_date,
  date_format(refund_timestamp, 'HH:mm:ss') AS refund_time,
  refund_amount,
  regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
  regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) AS refund_source
FROM
  awssql_gizmobox_refund_db_sql_catalog.public.refunds;

In [0]:
%sql
create table if not exists gizmobox.silver.refunds as
SELECT
  refund_id,
  payment_id,
  DATE(refund_timestamp) AS refund_date,
  date_format(refund_timestamp, 'HH:mm:ss') AS refund_time,
  refund_amount,
  regexp_extract(refund_reason, '^([^:]+):', 1) AS refund_reason,
  regexp_extract(refund_reason, '^[^:]+:(.*)$', 1) AS refund_source
FROM
  awssql_gizmobox_refund_db_sql_catalog.public.refunds;

In [0]:
%sql
select * from gizmobox.silver.refunds

In [0]:
%sql
desc extended gizmobox.silver.refunds